# Standing Goals — proactive-layer Analyse

Tier-1 eval (#389). Drives `GoalEvaluatorService` direkt (ohne Mission /
Agent / Executor). Fake-clock simuliert 60 Minuten, der cron-pre-filter
soll die Mehrheit der ticks LLM-frei halten (ADR-024).

## 1. Setup

In [ ]:
import os, sys, importlib
from pathlib import Path

PYTF_ROOT = Path.cwd()
if PYTF_ROOT.name == 'notebooks':
    PYTF_ROOT = PYTF_ROOT.parent
os.chdir(PYTF_ROOT)
for p in [str(PYTF_ROOT / 'src'), str(PYTF_ROOT / 'notebooks')]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from dotenv import load_dotenv
    load_dotenv(PYTF_ROOT / '.env')
except ImportError:
    pass

try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

import logging, structlog
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s | %(message)s')
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

import analysis_lib
importlib.reload(analysis_lib)
import analysis_lib as alib
print('analysis_lib OK -', len(alib.list_evaluators()), 'evaluators')

## 2. GoalEvaluatorService direkt aufsetzen

Standing-Goals laufen ohne Mission — die Evaluation triggert auf cron-fire.
Wir nutzen einen fake-clock + fake-decide um deterministisch zu testen, dass
der cron-Pre-Filter LLM-Calls einspart (ADR-024).

In [ ]:
from datetime import UTC, datetime, timedelta
from taskforce.core.domain.standing_goal import StandingGoal
from taskforce.core.domain.request import AgentRequest
from taskforce.application.goal_evaluator_service import (
    GoalEvaluatorService, GoalDecision,
)
from taskforce.infrastructure.persistence.file_standing_goal_store import (
    FileStandingGoalStore,
)

WORKDIR = Path('.taskforce_standing_goals_analysis')
WORKDIR.mkdir(exist_ok=True)
for f in WORKDIR.glob('*'):
    if f.is_file():
        f.unlink()
store = FileStandingGoalStore(work_dir=str(WORKDIR))

# Counters - the harness updates these via the fake hooks
submit_calls = 0
decide_calls = 0

async def fake_submit(request):
    global submit_calls
    submit_calls += 1

async def fake_decide(goal, now):
    global decide_calls
    decide_calls += 1
    # Always decides 'act' so we can verify submit count downstream
    return GoalDecision(act=True, mission='nb-test', rationale='fake')

current_time = [datetime(2026, 5, 17, 8, 0, tzinfo=UTC)]
def fake_clock():
    return current_time[0]

service = GoalEvaluatorService(
    store=store, submit=fake_submit, decide=fake_decide, clock=fake_clock,
)
print('service ready')

## 3. Goals registrieren + 60 Ticks à 1 Minute simulieren

In [ ]:
import asyncio

# Goal A: feuert nur Montag 9:00 → in unserem Window NIE due
await store.add(StandingGoal(
    description='Wochenrückblick',
    evaluation_prompt='ignored',
    frequency='0 9 * * 1',
))
# Goal B: feuert alle 15 Minuten → in 60 Min 4x due
await store.add(StandingGoal(
    description='Pulse-Check',
    evaluation_prompt='ignored',
    frequency='*/15 * * * *',
))

ticks = 0
decisions_per_tick = []
for _ in range(60):
    results = await service.evaluate_due_goals()
    ticks += 1
    decisions_per_tick.append(len(results))
    current_time[0] += timedelta(minutes=1)

print(f'ticks: {ticks}')
print(f'total decide() calls: {decide_calls}')
print(f'total submit() calls: {submit_calls}')
print(f'ticks with >=1 decision: {sum(1 for d in decisions_per_tick if d > 0)}')
print(f'decisions distribution: {dict((k, decisions_per_tick.count(k)) for k in set(decisions_per_tick))}')

## 4. Auswertung via `expected.standing_goals:` Evaluator

Wir konstruieren ein synthetisches `RunRecord` mit den gemessenen Werten
und scoren es gegen `notebooks/scenarios/standing_goals.yaml`.

In [ ]:
rec = alib.RunRecord()
rec.llm_calls = decide_calls   # decide() == LLM in real impl
rec.extra['heartbeat_ticks'] = ticks
rec.extra['standing_goals_decisions'] = decide_calls

scenarios = alib.load_scenarios('notebooks/scenarios/standing_goals.yaml')
eligible = scenarios   # no requires gating here
AVAILABLE_TOOLS = set()  # not used for SG
print(f'{len(eligible)} scenarios')
for s in eligible:
    card = alib.score_rule_based(rec, s)
    mark = 'PASS' if card.passed else 'FAIL'
    print(f'  [{mark}] {s.id}')
    alib.print_feature_checks(card)
    if card.details:
        for d in card.details:
            print(f'    - {d}')

## 5. Pre-Filter-Effizienz visualisieren

Der Punkt von ADR-024: cron-pre-filter sollte den Großteil der ticks OHNE
LLM-Call abdecken. Die untere Zahl ist die kritische Metrik.

In [ ]:
print(f'LLM-call ratio: {decide_calls / ticks:.3f} (lower = better; ADR-024 cron pre-filter)')
print(f'Expected upper bound for 60 min window with 15-min cron + weekly cron: ~0.07')


## Ideen für weitere Experimente

Siehe TODOs in der Scenario-YAML.

In [ ]:
print('done')